https://github.com/CyanHuynh10/AI_EXERCISE

### 1. Phân tích P.E.A.S
* **P (Performance Measure):** Chỉ số hiệu suất dựa trên độ sạch của môi trường và tổng chi phí của các hành động. Thuật toán UCS luôn ưu tiên mở rộng các đường đi có chi phí thấp nhất, đảm bảo tìm được chuỗi hành động làm sạch môi trường với chi phí tối ưu.
* **E (Environment):** Một lưới ô vuông 3x3. Mỗi ô có thể ở trạng thái "Sạch" (Clean) hoặc "Bẩn" (Dirty). Môi trường có thể quan sát toàn phần (Fully Observable) để thuật toán duyệt và tính toán chi phí trên toàn bản đồ.
* **A (Actuators):** Các bộ phận thực hiện hành động: Di chuyển (UP, DOWN, LEFT, RIGHT), Hút bụi (SUCK), và Dừng (STOP). Mỗi hành động đều có một mức chi phí (thường giả định chi phí = 1 cho mỗi hành động).
* **S (Sensors):** Cảm biến vị trí (biết Agent đang ở ô nào) và cảm biến bụi (nhận biết toàn bộ vị trí các ô bẩn trên lưới).

### 2. Các thành phần của Agent theo kiến trúc Model-Based
* **State (Trạng thái hiện tại):** Là sự kết hợp giữa tọa độ hiện tại của Agent và danh sách các ô còn bẩn trên lưới 3x3.
* **Percept:** Dữ liệu nhận vào gồm vị trí hiện tại và trạng thái của toàn bộ lưới môi trường.
* **Model:**
    * **Hiểu biết về môi trường:** Nắm được quy luật dịch chuyển và tương tác, đồng thời biết được chi phí của mỗi hành động để cộng dồn vào tổng chi phí đường đi (Path Cost).
    * **Lưu trữ thông tin (Internal State):** Trạng thái đích (Goal), Hàng đợi ưu tiên (Priority Queue) sắp xếp theo chi phí để phục vụ thuật toán UCS, tập các trạng thái đã duyệt (Visited), và Kế hoạch (Plan) lưu chuỗi hành động tối ưu chưa được thực thi.
* **Rules (Tập luật và Lập luận IF):**
    * `IF` môi trường đã sạch hoàn toàn (Không còn ô bẩn) `THEN` Action = "STOP".
    * `IF` chưa đạt đích VÀ danh sách Plan đang trống `THEN` Kích hoạt tìm kiếm UCS để tạo chuỗi hành động có chi phí thấp nhất làm sạch tất cả ô bẩn -> Cập nhật vào Plan -> Action = Plan.pop(0).
    * `IF` danh sách Plan đã có sẵn các bước đi `THEN` Action = Plan.pop(0).

### 3. Trạng thái môi trường (GOAL STATE)
Trạng thái đích đạt được khi tất cả các ô trong lưới môi trường đều chuyển về trạng thái **Clean**.
Ví dụ với lưới 3x3:
- (0,0): Clean
- (0,1): Clean
- (0,2): Clean
- (1,0): Clean
- (1,1): Clean
- (1,2): Clean
- (2,0): Clean
- (2,1): Clean
- (2,2): Clean

In [2]:
import copy
import heapq

def ucs_search(start_state):
    count = 0
    pq = [(0, count, start_state, [])]
    visited = set()

    while pq:
        cost, _, current_state, path = heapq.heappop(pq)
        r, c, dirty = current_state
        
        if len(dirty) == 0:
            return path
            
        if current_state in visited:
            continue
        visited.add(current_state)
        
        successors = []
        if (r, c) in dirty:
            new_dirty = frozenset(d for d in dirty if d != (r, c))
            successors.append((cost + 1, (r, c, new_dirty), 'SUCK'))
        else:
            if r > 0: successors.append((cost + 1, (r-1, c, dirty), 'UP'))
            if r < 2: successors.append((cost + 1, (r+1, c, dirty), 'DOWN'))
            if c > 0: successors.append((cost + 1, (r, c-1, dirty), 'LEFT'))
            if c < 2: successors.append((cost + 1, (r, c+1, dirty), 'RIGHT'))
            
        for next_cost, next_state, action in successors:
            if next_state not in visited:
                count += 1
                heapq.heappush(pq, (next_cost, count, next_state, path + [action]))
    return []

class VacuumEnvironment:
    def __init__(self, grid_size=(3, 3), dirty_cells=None):
        self.grid_size = grid_size
        self.grid = { (r, c): 'Clean' for r in range(grid_size[0]) for c in range(grid_size[1]) }
        if dirty_cells:
            for cell in dirty_cells:
                self.grid[cell] = 'Dirty'

    def get_percept(self, agent_pos):
        return {'pos': agent_pos, 'grid': copy.deepcopy(self.grid)}

    def execute_action(self, agent_pos, action):
        if action == 'SUCK':
            self.grid[agent_pos] = 'Clean'
        elif action == 'UP' and agent_pos[0] > 0:
            agent_pos = (agent_pos[0] - 1, agent_pos[1])
        elif action == 'DOWN' and agent_pos[0] < self.grid_size[0] - 1:
            agent_pos = (agent_pos[0] + 1, agent_pos[1])
        elif action == 'LEFT' and agent_pos[1] > 0:
            agent_pos = (agent_pos[0], agent_pos[1] - 1)
        elif action == 'RIGHT' and agent_pos[1] < self.grid_size[1] - 1:
            agent_pos = (agent_pos[0], agent_pos[1] + 1)
        return agent_pos

class ModelBasedVacuumAgent:
    def __init__(self):
        self.plan = []
        self.plan_count = 0

    def get_heuristic(self, grid):
        return sum(1 for status in grid.values() if status == 'Dirty')

    def agent_function(self, percept):
        pos = percept['pos']
        grid = percept['grid']
        
        dirty_cells = frozenset([k for k, v in grid.items() if v == 'Dirty'])
        
        if len(dirty_cells) == 0:
            return 'STOP'
            
        if not self.plan:
            start_state = (pos[0], pos[1], dirty_cells)
            self.plan = ucs_search(start_state)
            self.plan_count += 1
            
        if self.plan:
            return self.plan.pop(0)
        return 'STOP'

def print_board(grid, agent_pos):
    print("+-----+-----+-----+")
    for r in range(3):
        row_str = "|"
        for c in range(3):
            status = 'D' if grid[(r, c)] == 'Dirty' else 'C'
            if (r, c) == agent_pos:
                val_str = f" [{status}] " 
            else:
                val_str = f"  {status}  "
            row_str += val_str + "|"
        print(row_str)
        print("+-----+-----+-----+")

def run_simulation():
    env = VacuumEnvironment(grid_size=(3, 3), dirty_cells=[(0, 1), (1, 2), (2, 0)])
    agent = ModelBasedVacuumAgent()
    agent_pos = (0, 0)
    step = 0
    
    percept = env.get_percept(agent_pos)
    initial_h = agent.get_heuristic(percept['grid'])

    print("=========================================================")
    print(f"TRẠNG THÁI BAN ĐẦU (Step: {step} | Heuristic: {initial_h})")
    print("=========================================================")
    print_board(env.grid, agent_pos)

    while True:
        step += 1
        percept = env.get_percept(agent_pos)
        action = agent.agent_function(percept)
        
        if action == 'STOP':
            print("=========================================================")
            print(f"KẾT QUẢ: MÔI TRƯỜNG ĐÃ SẠCH HOÀN TOÀN TRONG {step - 1} BƯỚC!")
            print(f"Số lần agent phải lập Plan: {agent.plan_count}")
            print("=========================================================")
            break
            
        agent_pos = env.execute_action(agent_pos, action)
        h = agent.get_heuristic(env.grid)
        
        print("---------------------------------------------------------")
        print(f"Step: {step:02d} | Action: {action} | Heuristic: {h}")
        print_board(env.grid, agent_pos)
        
        if step > 50:
            break

run_simulation()

TRẠNG THÁI BAN ĐẦU (Step: 0 | Heuristic: 3)
+-----+-----+-----+
| [C] |  D  |  C  |
+-----+-----+-----+
|  C  |  C  |  D  |
+-----+-----+-----+
|  D  |  C  |  C  |
+-----+-----+-----+
---------------------------------------------------------
Step: 01 | Action: RIGHT | Heuristic: 3
+-----+-----+-----+
|  C  | [D] |  C  |
+-----+-----+-----+
|  C  |  C  |  D  |
+-----+-----+-----+
|  D  |  C  |  C  |
+-----+-----+-----+
---------------------------------------------------------
Step: 02 | Action: SUCK | Heuristic: 2
+-----+-----+-----+
|  C  | [C] |  C  |
+-----+-----+-----+
|  C  |  C  |  D  |
+-----+-----+-----+
|  D  |  C  |  C  |
+-----+-----+-----+
---------------------------------------------------------
Step: 03 | Action: DOWN | Heuristic: 2
+-----+-----+-----+
|  C  |  C  |  C  |
+-----+-----+-----+
|  C  | [C] |  D  |
+-----+-----+-----+
|  D  |  C  |  C  |
+-----+-----+-----+
---------------------------------------------------------
Step: 04 | Action: RIGHT | Heuristic: 2
+-----